<h1 style = "color:blue">Urban Waterlogging Risk Prediction</h1>

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
N_ROWS = 5000
rainfall_intensity = np.random.gamma(shape=2.0, scale=8.0, size=N_ROWS)
rainfall_intensity = np.clip(rainfall_intensity, 0, 200)
rainfall_duration = np.random.gamma(shape=2.0, scale=1.5, size=N_ROWS)
rainfall_duration = np.clip(rainfall_duration, 0.25, 48)
# accumulated rainfall in the prior 48 hours
previous_rainfall = np.random.exponential(scale=20, size=N_ROWS)
previous_rainfall = np.clip(previous_rainfall, 0, 200)
elevation = np.random.normal(loc=45, scale=15, size=N_ROWS)
elevation = np.clip(elevation, 15, 100)
drainage_capacity = np.random.normal(loc=35, scale=15, size=N_ROWS)
drainage_capacity = np.clip(drainage_capacity, 5, 100)
impervious_surface = np.random.beta(a=2.0, b=1.5, size=N_ROWS) * 100
impervious_surface = np.clip(impervious_surface, 10, 98)

In [2]:
water_load = (
    (rainfall_intensity * 0.6) +
    (rainfall_duration * 4.0) +
    (previous_rainfall * 0.3)
)
coping_capacity = drainage_capacity * 1.2
lowness_factor = 1 + ((100 - elevation) / 100)*0.8
imperviousness_factor = 1 + (impervious_surface / 100)+1.0
raw_risk_score = (water_load * lowness_factor * imperviousness_factor) - coping_capacity
noice = np.random.normal(loc=0, scale=5, size = N_ROWS)
risk_score = raw_risk_score + noice

In [3]:
labels = np.array(["LOW", "MODERATE", "HIGH", "CRITICAL"])
q25, q50, q75 = np.percentile(risk_score, [25, 50, 75])
risk_level = labels[np.digitize(risk_score, [q25, q50, q75])]

In [4]:
df = pd.DataFrame({
    "rainfall_intensity": rainfall_intensity,
    "rainfall_duration": rainfall_duration,
    "elevation": elevation,
    "drainage_capacity": drainage_capacity,
    "previous_rainfall": previous_rainfall,
    "impervious_surface": impervious_surface,
    "risk_level": risk_level
})
df.head()

,rainfall_intensity,rainfall_duration,elevation,drainage_capacity,previous_rainfall,impervious_surface,risk_level
0,19.149435,2.235583,24.384011,43.793613,18.718302,84.092982,HIGH
1,11.955718,3.132334,18.626760,64.469184,30.382600,97.630082,HIGH
2,11.058269,8.876425,38.884418,38.830987,11.951820,25.434052,CRITICAL
3,11.058418,8.682093,43.340714,44.620748,15.186051,16.623043,CRITICAL
4,37.197715,4.107852,71.468027,17.519973,11.205714,83.395811,CRITICAL


In [5]:
df["risk_level"].value_counts()

risk_level
HIGH        1250
CRITICAL    1250
LOW         1250
MODERATE    1250
Name: count, dtype: int64

<h3 style = "color : green">Rounding & Suffling the data</h3>

In [10]:
df["rainfall_intensity"] = np.round(df["rainfall_intensity"], 2)
df["rainfall_duration"] = np.round(df["rainfall_duration"], 2)
df["elevation"] = np.round(df["elevation"], 2)
df["drainage_capacity"] = np.round(df["drainage_capacity"], 2)
df["previous_rainfall"] = np.round(df["previous_rainfall"], 2)
df["impervious_surface"] = np.round(df["impervious_surface"], 2)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv("waterlogging_data.csv", index=False)
df.head()

,rainfall_intensity,rainfall_duration,elevation,drainage_capacity,previous_rainfall,impervious_surface,risk_level,net_water_balance
0,16.31,1.91,34.67,42.27,24.52,36.89,MODERATE,-1.44
1,18.35,1.38,49.91,30.45,15.41,16.05,MODERATE,3.31
2,29.73,3.89,59.68,40.49,3.60,58.94,HIGH,-7.16
3,32.72,2.05,28.42,27.99,31.77,81.21,CRITICAL,36.50
4,4.99,3.48,45.39,31.05,9.10,67.33,MODERATE,-16.96


<h3 style = "color : green">read the csv file and separating features and target</h3>

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("waterlogging_data.csv")
df["net_water_balance"] = (df["rainfall_intensity"] + df["previous_rainfall"]) - df["drainage_capacity"]
FEATURE_COLUMNS = [
    "rainfall_intensity",
    "rainfall_duration",
    "elevation",
    "drainage_capacity",
    "previous_rainfall",
    "impervious_surface",
    "net_water_balance"
]
X = df[FEATURE_COLUMNS]
y = df["risk_level"]
df[["rainfall_intensity", "previous_rainfall", "drainage_capacity", "net_water_balance"]].head()

,rainfall_intensity,previous_rainfall,drainage_capacity,net_water_balance
0,16.31,24.52,42.27,-1.44
1,18.35,15.41,30.45,3.31
2,29.73,3.60,40.49,-7.16
3,32.72,31.77,27.99,36.50
4,4.99,9.10,31.05,-16.96


In [12]:
X.head()

,rainfall_intensity,rainfall_duration,elevation,drainage_capacity,previous_rainfall,impervious_surface,net_water_balance
0,16.31,1.91,34.67,42.27,24.52,36.89,-1.44
1,18.35,1.38,49.91,30.45,15.41,16.05,3.31
2,29.73,3.89,59.68,40.49,3.60,58.94,-7.16
3,32.72,2.05,28.42,27.99,31.77,81.21,36.50
4,4.99,3.48,45.39,31.05,9.10,67.33,-16.96


In [13]:
y.head()

0    MODERATE
1    MODERATE
2        HIGH
3    CRITICAL
4    MODERATE
Name: risk_level, dtype: str

In [14]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Label mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {i}")

Label mapping:
  CRITICAL -> 0
  HIGH -> 1
  LOW -> 2
  MODERATE -> 3


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
print(f"train size: {X_train.shape[0]}, test size: {X_test.shape[0]}")

train size: 4000, test size: 1000


<h3 style = "color : green">Features Scaling</h3>

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

<h3 style = "color : green">SVM Model</h3>

In [17]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=42,
    probability=True
)
svm_model.fit(X_train_scaled, y_train)

C:\Users\KIIT\anaconda3\envs\urbanWater\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [18]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = svm_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

accuracy: 0.9020

Classification Report:
              precision    recall  f1-score   support

    CRITICAL       0.94      0.94      0.94       250
        HIGH       0.88      0.88      0.88       250
         LOW       0.95      0.91      0.93       250
    MODERATE       0.85      0.88      0.87       250

    accuracy                           0.90      1000
   macro avg       0.90      0.90      0.90      1000
weighted avg       0.90      0.90      0.90      1000


Confusion Matrix:
[[236  14   0   0]
 [ 16 219   0  15]
 [  0   0 227  23]
 [  0  17  13 220]]


In [19]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores = cross_val_score(svm_pipeline, X_train, y_train, cv=cv, scoring="accuracy")
f1_scores = cross_val_score(svm_pipeline, X_train, y_train, cv=cv, scoring="f1_macro")
print("Accuracy per fold:", acc_scores.round(3))
print(f"Mean accuracy: {acc_scores.mean():.4f} (+/- {acc_scores.std():.4f})")
print(f"Mean macro F1: {f1_scores.mean():.4f} (+/- {f1_scores.std():.4f})")

Accuracy per fold: [0.892 0.9   0.869 0.889 0.886]
Mean accuracy: 0.8872 (+/- 0.0104)
Mean macro F1: 0.8878 (+/- 0.0100)


In [20]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1]
}
grid_svm = GridSearchCV(
    svm_pipeline,
    param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)
grid_svm.fit(X_train, y_train)
print("Best parameters:", grid_svm.best_params_)
print(f"Best CV macro F1: {grid_svm.best_score_:.4f}")

Best parameters: {'svm__C': 10, 'svm__gamma': 0.01}
Best CV macro F1: 0.9020


In [21]:
y_pred_tuned = grid_svm.predict(X_test)
print(f"Tuned test accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print("\nTuned Classification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=label_encoder.classes_))
print("\nTuned Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Tuned test accuracy: 0.9110

Tuned Classification Report:
              precision    recall  f1-score   support

    CRITICAL       0.95      0.95      0.95       250
        HIGH       0.90      0.89      0.89       250
         LOW       0.94      0.92      0.93       250
    MODERATE       0.86      0.88      0.87       250

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000


Tuned Confusion Matrix:
[[238  12   0   0]
 [ 12 222   0  16]
 [  0   0 230  20]
 [  0  14  15 221]]


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("rf", RandomForestClassifier(random_state=42))
])
rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring="f1_macro")
print("Macro F1 per fold:", rf_scores.round(3))
print(f"Mean macro F1: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")

Macro F1 per fold: [0.831 0.832 0.825 0.826 0.839]
Mean macro F1: 0.8305 (+/- 0.0050)


In [23]:
rf_param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_split": [2, 5, 10]
}
grid_rf = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)
print("Best parameters:", grid_rf.best_params_)
print(f"Best CV macro F1: {grid_rf.best_score_:.4f}")

Best parameters: {'rf__max_depth': None, 'rf__min_samples_split': 5, 'rf__n_estimators': 200}
Best CV macro F1: 0.8391


In [24]:
y_pred_rf = grid_rf.predict(X_test)
print(f"Tuned RF test accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nTuned RF Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=label_encoder.classes_))
print("\nTuned RF Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Tuned RF test accuracy: 0.8350

Tuned RF Classification Report:
              precision    recall  f1-score   support

    CRITICAL       0.87      0.89      0.88       250
        HIGH       0.77      0.78      0.77       250
         LOW       0.92      0.88      0.90       250
    MODERATE       0.79      0.79      0.79       250

    accuracy                           0.83      1000
   macro avg       0.84      0.83      0.84      1000
weighted avg       0.84      0.83      0.84      1000


Tuned RF Confusion Matrix:
[[222  27   0   1]
 [ 32 195   0  23]
 [  0   0 220  30]
 [  0  32  20 198]]


In [25]:
comparison = pd.DataFrame({
    "Model": ["Tuned SVM", "Tuned Random Forest"],
    "CV Macro F1": [grid_svm.best_score_, grid_rf.best_score_],
    "Test Accuracy": [
        accuracy_score(y_test, grid_svm.predict(X_test)),
        accuracy_score(y_test, y_pred_rf)
    ]
})
print(comparison)

                 Model  CV Macro F1  Test Accuracy
0            Tuned SVM     0.901999          0.911
1  Tuned Random Forest     0.839100          0.835


<h3 style = "color : green">JOBLIB --- SAVE THE PYTHON FILE</h3>

In [26]:
import joblib

joblib.dump(grid_svm.best_estimator_, "final_svm_pipeline.joblib")
joblib.dump(grid_rf.best_estimator_, "final_rf_pipeline.joblib")
joblib.dump(label_encoder, "label_encoder.joblib")

['label_encoder.joblib']